Tugas Kelompok ke-2 Week 8
OPTIMASI DAN IMPLEMENTASI DALAM APLIKASI SEDERHANA (LO 3, Bobot 100)

Group 2
1. INDRA KOESUMAH - 2902742483
2. INDHAH PUJIHASTUTI - 2902712133
3. ALVIYAN SYAFRIANSAH MATONDANG - 2902730565
4. INDRI TALITHA - 2902743100
5. MUHAMAD DEKY AKBAR - 2902727854

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Import library TensorFlow dan Keras
import tensorflow as tf
from tensorflow.keras import layers, models

# Load dataset CIFAR-10 (10 kelas gambar)
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Normalisasi pixel (0-255 → 0-1)
train_images = train_images / 255.0
test_images = test_images / 255.0

# Membuat model CNN (dengan optimasi)
model = models.Sequential([

    # Input layer (fix warning Keras)
    layers.Input(shape=(32,32,3)),

    # Convolution layer 1
    layers.Conv2D(32, (3,3), activation='relu'),

    # Batch Normalization → mempercepat & stabil
    layers.BatchNormalization(),

    # Pooling → mengurangi dimensi
    layers.MaxPooling2D(),

    # Convolution layer 2
    layers.Conv2D(64, (3,3), activation='relu'),

    # Dropout → mencegah overfitting
    layers.Dropout(0.3),

    layers.MaxPooling2D(),

    # Flatten → ubah ke 1D
    layers.Flatten(),

    # Dense layer
    layers.Dense(64, activation='relu'),

    # Output layer (10 kelas)
    layers.Dense(10, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training model
model.fit(
    train_images,
    train_labels,
    epochs=5,
    validation_data=(test_images, test_labels)
)

# simpan model format .keras (bukan .h5)
model.save("model.keras")

print("Model berhasil disimpan dalam format .keras!")

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 96s 60ms/step - accuracy: 0.5239 - loss: 1.3390 - val_accuracy: 0.5461 - val_loss: 1.2666
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 86s 55ms/step - accuracy: 0.6455 - loss: 1.0121 - val_accuracy: 0.6597 - val_loss: 0.9848
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 86s 55ms/step - accuracy: 0.6856 - loss: 0.9018 - val_accuracy: 0.6114 - val_loss: 1.1290
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 143s 56ms/step - accuracy: 0.7089 - loss: 0.8353 - val_accuracy: 0.6367 - val_loss: 1.0567
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.7270 - loss: 0.7817 - val_accuracy: 0.6800 - val_loss: 0.9264
Model berhasil disimpan dalam format .keras!


In [14]:
import os

# Define the directory path
drive_path = '/content/drive/MyDrive/AI-Project'

# Create the directory if it doesn't exist
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"Directory '{drive_path}' created successfully.")
else:
    print(f"Directory '{drive_path}' already exists.")

Directory '/content/drive/MyDrive/AI-Project' already exists.


In [15]:
model.save('/content/drive/MyDrive/AI-Project/model.keras')
print("Model saved to Google Drive successfully!")

Model saved to Google Drive successfully!


In [10]:
# install server & tunneling
!pip install fastapi uvicorn pyngrok pillow numpy

### Set your ngrok authtoken

To get your ngrok authtoken:
1. Go to the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
2. Copy your authtoken.
3. Replace `YOUR_AUTHTOKEN` in the code below with your actual authtoken and run the cell.

In [16]:
from pyngrok import ngrok

# Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken
ngrok.set_auth_token("YOUR_AUTHTOKEN")

In [26]:
# load model
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model("/content/drive/MyDrive/AI-Project/model.keras")

# class CIFAR-10
class_names = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]


In [25]:
# bikin API
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import io

app = FastAPI()

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB").resize((32,32))

    img = np.array(image)/255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)
    idx = int(np.argmax(preds))
    conf = float(np.max(preds))

    # 🔥 FILTER BIAR GAK NGACO
    if conf < 0.6:
      return {
        "class": "Unknown",
        "confidence": conf
      }

    return {
      "class": class_names[idx],
      "confidence": conf
    }

In [20]:
from pyngrok import ngrok

ngrok.set_auth_token("3D4czGTlsucEca5u1CR6QpkOU5P_4tuUMEP7FwV47hqyG934g")

In [21]:
# expose ke internet (link publik)
from pyngrok import ngrok
import uvicorn
import threading

public_url = ngrok.connect(8000)
print("API URL:", public_url)

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

API URL: NgrokTunnel: "https://excluding-decaf-bruising.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [7805]
INFO:     Waiting for application startup.
